# STTran · 300 videos · Action Genome (zipped annotations + frames)

This notebook **downloads your repo on Colab** (GitHub zip first — no `git` username prompt), downloads **GloVe + Faster R-CNN + STTran weights**, merges **`annotations.zip`** and **`frames.zip`** into a valid AG tree, then runs **`run_first5_videos_all_frames.py`** with **`VIDEO_LIMIT=300`**.

**Outputs** (under `STTran/output/`):
- `logs/first5_videos/<VIDEO>.mp4.log` — full terminal-style log (parse for graphs / stats).
- `first5_videos/<VIDEO>.mp4/` — per-frame PNGs, GIF, legends.

After the run, **`export_graphs_json.py`** writes **`graphs.json`** in each video folder for programmatic graph extraction.

Push this repo (including `colab_minimal/`) to **GIT_REPO**, then edit the **config cell** paths (`GIT_REPO`, Drive zip paths). Use a **GPU** runtime before running.



In [ ]:
# --- EDIT THESE ---
GIT_REPO = "https://github.com/TommasoAiello08/STTran.git"
GIT_BRANCH = "main"  # change if your default branch differs

# Two zips on Google Drive (paths must exist after mounting Drive):
ANNOTATIONS_ZIP = "/content/drive/MyDrive/AG_zips/annotations.zip"
FRAMES_ZIP = "/content/drive/MyDrive/AG_zips/frames.zip"

# Where to build the Action Genome tree (ephemeral local disk on Colab):
AG_ROOT = "/content/ag_data"

# STTran code + outputs 
REPO_ROOT = "/content/STTran"

# How many videos to process (first N in frame_list order that appear in the chosen split)
VIDEO_LIMIT = 300
SPLIT = "test"   # "train" or "test"

# Optional: zip results back to Drive when done
OUTPUT_ZIP_ON_DRIVE = "/content/drive/MyDrive/sttran_outputs_300.zip"


In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)


In [ ]:
import os, re, shutil, subprocess, sys, tempfile, urllib.request, zipfile
from pathlib import Path


def run(cmd, cwd=None, env=None):
    print("+", " ".join(cmd), flush=True)
    subprocess.check_call(cmd, cwd=cwd, env=env)


def _rm_repo():
    p = Path(REPO_ROOT)
    if p.exists():
        shutil.rmtree(p, ignore_errors=True)
    subprocess.run(["rm", "-rf", REPO_ROOT], capture_output=True)


def _git_env():
    e = os.environ.copy()
    e["GIT_TERMINAL_PROMPT"] = "0"
    e.pop("GIT_ASKPASS", None)
    return e


def _github_slug(url):
    m = re.search(r"github\.com[:/]([^/]+)/([^/\s]+?)(?:\.git)?(?:/|\?|$)", url)
    if not m:
        return None
    return f"{m.group(1)}/{m.group(2)}"


def _http_get_bin(url: str) -> bytes:
    """GitHub often rejects urllib default User-Agent; use a normal browser UA."""
    req = urllib.request.Request(
        url,
        headers={
            "User-Agent": "Mozilla/5.0 (compatible; Colab-STTran/1; +https://colab.research.google.com/)"
        },
    )
    with urllib.request.urlopen(req, timeout=120) as resp:
        return resp.read()


def _download_zipball() -> bool:
    """Fetch public repo as zip — never invokes git (avoids Colab username prompt entirely)."""
    slug = _github_slug(GIT_REPO)
    if not slug:
        print("Could not parse owner/repo from GIT_REPO:", GIT_REPO)
        return False
    _rm_repo()
    urls = [
        f"https://codeload.github.com/{slug}/zip/refs/heads/{GIT_BRANCH}",
        f"https://github.com/{slug}/archive/refs/heads/{GIT_BRANCH}.zip",
        f"https://github.com/{slug}/archive/{GIT_BRANCH}.zip",
    ]
    tmp = Path(tempfile.mkdtemp())
    zpath = tmp / "src.zip"
    last_err = None
    for url in urls:
        try:
            print("Downloading:", url, flush=True)
            data = _http_get_bin(url)
            zpath.write_bytes(data)
            if zpath.stat().st_size < 2000:
                print("  response too small; trying next mirror")
                continue
            break
        except Exception as ex:
            last_err = ex
            print("  failed:", ex)
    else:
        shutil.rmtree(tmp, ignore_errors=True)
        print("All zip URLs failed. Last error:", last_err)
        return False
    with zipfile.ZipFile(zpath, "r") as zf:
        zf.extractall(tmp)
    top = [p for p in tmp.iterdir() if p.is_dir() and p.name not in ("__MACOSX",)]
    if len(top) != 1:
        print("Unexpected archive layout:", [p.name for p in tmp.iterdir()])
        shutil.rmtree(tmp, ignore_errors=True)
        return False
    shutil.move(str(top[0]), REPO_ROOT)
    shutil.rmtree(tmp, ignore_errors=True)
    print("OK: GitHub zipball ->", REPO_ROOT)
    return True


def _git_clone_fallback():
    """Only if zip failed — last resort on Colab."""
    ge = _git_env()
    cmd_branch = [
        "git",
        "-c",
        "credential.helper=",
        "clone",
        "--depth",
        "1",
        "-b",
        GIT_BRANCH,
        GIT_REPO,
        REPO_ROOT,
    ]
    print("+", " ".join(cmd_branch), flush=True)
    r = subprocess.run(cmd_branch, capture_output=True, text=True, env=ge)
    if r.returncode == 0:
        return True
    print("Git stderr:", r.stderr or r.stdout or "(empty)")
    _rm_repo()
    cmd_def = [
        "git",
        "-c",
        "credential.helper=",
        "clone",
        "--depth",
        "1",
        GIT_REPO,
        REPO_ROOT,
    ]
    print("+", " ".join(cmd_def), flush=True)
    r2 = subprocess.run(cmd_def, capture_output=True, text=True, env=ge)
    if r2.returncode == 0:
        return True
    print("Git stderr:", r2.stderr or r2.stdout or "(empty)")
    return False


_rm_repo()

# Colab: zip FIRST so git never prompts "could not read Username" for public repos.
if not _download_zipball():
    print("Zipball download failed — trying git clone as last resort.")
    if not _git_clone_fallback():
        raise RuntimeError(
            "Could not fetch repo. Public: check GIT_REPO / GIT_BRANCH. "
            "Private: use GIT_REPO = https://<TOKEN>@github.com/OWNER/REPO.git"
        )

if (Path(REPO_ROOT) / ".git").is_dir():
    print(subprocess.check_output(["git", "log", "-1", "--oneline"], cwd=REPO_ROOT).decode())
else:
    print("(fetched via zip — no .git; source tree is complete for running STTran)")

os.chdir(REPO_ROOT)
print("REPO_ROOT:", REPO_ROOT)




In [ ]:
# Merge annotations.zip + frames.zip -> AG_ROOT/{annotations,frames}
from pathlib import Path
import importlib.util

ann = Path(ANNOTATIONS_ZIP)
frm = Path(FRAMES_ZIP)
assert ann.is_file(), f"Missing {ann}"
assert frm.is_file(), f"Missing {frm}"

spec = importlib.util.spec_from_file_location(
    "prepare_ag_zips",
    Path(REPO_ROOT) / "colab_minimal" / "prepare_ag_zips.py",
)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

mod.prepare_action_genome(ann, frm, Path(AG_ROOT), clean=True)
print("AG_ROOT ready:", AG_ROOT)


In [ ]:
# Install deps, compile extensions, GloVe, download official STTran + Faster R-CNN weights
run([sys.executable, "setup_colab.py"], cwd=REPO_ROOT)


In [ ]:
import os
os.environ["AG_DATA_PATH"] = AG_ROOT
os.environ["STTRAN_CKPT"] = str(Path(REPO_ROOT) / "ckpts" / "sttran_predcls.tar")
os.environ["SPLIT"] = SPLIT
os.environ["VIDEO_LIMIT"] = str(VIDEO_LIMIT)
os.environ["TOPK_PER_GROUP"] = "4"
os.environ["EDGE_THRESH"] = "0.0"
os.environ["VIZ_LAYOUT"] = "circular"
os.environ["VIZ_REUSE_LAYOUT"] = "1"
os.environ["MAX_RELS"] = "2000"

for k in ("AG_DATA_PATH", "VIDEO_LIMIT", "SPLIT"):
    print(k, os.environ.get(k))


In [ ]:
# Main run (~long): one folder per video under output/first5_videos/<VIDEO>.mp4/
run([sys.executable, "-u", "run_first5_videos_all_frames.py"], cwd=REPO_ROOT)


In [ ]:
# Parse logs -> graphs.json in each video folder (for downstream analysis)
import os
from pathlib import Path
os.environ.setdefault("TOPK_SPATIAL", "4")
os.environ.setdefault("TOPK_CONTACT", "4")
run([sys.executable, "export_graphs_json.py"], cwd=REPO_ROOT)


In [ ]:
# Optional: pack output/ into one zip on Drive (logs + viz + graphs.json)
from pathlib import Path
import shutil
out_dir = Path(REPO_ROOT) / "output"
if not out_dir.is_dir():
    print("No output directory — skipping zip.")
else:
    z = Path(OUTPUT_ZIP_ON_DRIVE)
    if z.suffix.lower() != ".zip":
        z = z.with_suffix(".zip")
    base = str(z.with_suffix(""))
    shutil.make_archive(base, "zip", root_dir=str(Path(REPO_ROOT)), base_dir="output")
    print("Wrote:", z)
